## Google Trends Time Series Collection

**Data collection pipeline for the event-word Google Trends corpus.**

This notebook documents, end to end, how the daily Google Trends
`interest_over_time` series were collected for the 1203 curated
event-related keywords used in this study, spanning `2022-01-01`
through `2026-05-31` (US geo). The data collection phase for this
project is complete, and the raw and processed outputs are provided
alongside this repository. See`src_SAX/google_trends_word_pull_original.ipynb`
for the source code.


## What this notebook shows

1. **Configuration** — every hardcoded parameter used during the
   original run (keyword source, date range, geo target, chunking
   scheme, and rate-limiting constants).
2. **Network layer** — `pytrends` wrapper implementing
   randomized adaptive pacing and exponential-style backoff, built
   around Google Trends' rate limits.
3. **Pipeline orchestration** — how 1203 keywords x multiple
   overlapping date windows were batched, checkpointed, and logged,
   so that a run could be killed and resumed without re-fetching
   completed work or corrupting partial output.
4. **Post-processing** — how overlapping chunks were stitched back
   into a single continuous daily series per keyword, including the
   spike-aware rescaling method that superseded a naive first pass
   (see the design note in Section 4).

## Why the code has this structure

Google Trends' public `interest_over_time` endpoint caps any single
query at roughly 270 days before it silently switches from daily to
weekly resolution, and it enforces an undocumented per-session rate
limit that responds with HTTP 429 once tripped. Both constraints
drove the design below: we pull keywords in overlapping ~270-day
chunks (Section 1), and separate every request by a randomized,
periodically-lengthened delay (Section 2).

## Output layout

Each keyword gets its own directory under `data/raw_pulls/`:

```
data/raw_pulls/<keyword_slug>/
    chunks/          one CSV per (keyword, date window) pull
    diagnostics/      per-chunk and per-stitch QA tables
    stitched/         the reassembled full-range daily series
    metadata.json     provenance: query text, source, run params
```


## 1. Global Configuration & Hyperparameters

Every path, date boundary, and rate-limiting constant used during
the original collection run lives here. Nothing below is inferred
or re-derived elsewhere in the notebook — Sections 2 through 4
import these names rather than re-declaring their own defaults.

Keyword lists are **not** inlined as string literals. The original
run sourced 847 curated event-related keywords (a `top` list and a
`rising` list, per Google Trends terminology) from two plain-text
files, one query per line. Reviewers who want to inspect the exact
keyword set should look at `data/keywords/`; the loading and
deduplication logic is in Section 3.


In [ ]:
"""Configuration and hyperparameters for the Google Trends pull.

This cell is the single source of truth for the original collection
run: paths, the Trends query window, chunking geometry, and every
constant that shaped the adaptive-pacing / backoff behavior in
Section 2. Downstream cells only ever read these names.
"""
from pathlib import Path

# --------------------------------------------------------------- #
# Project paths
# --------------------------------------------------------------- #
DATA_DIR: Path = Path("data")
KEYWORDS_DIR: Path = DATA_DIR / "keywords"
RAW_PULLS_DIR: Path = DATA_DIR / "raw_pulls"
LOG_DIR: Path = DATA_DIR / "logs"

# One curated query per line; see the Section 1 header note.
TOP_QUERIES_PATH: Path = KEYWORDS_DIR / "top_queries.txt"
RISING_QUERIES_PATH: Path = KEYWORDS_DIR / "rising_queries.txt"

RUN_LOG_PATH: Path = RAW_PULLS_DIR / "run_log.csv"
QUERY_METADATA_PATH: Path = RAW_PULLS_DIR / "query_metadata.csv"
SCALE_DIAGNOSTICS_PATH: Path = DATA_DIR / "scale_diagnostics.csv"

for directory in (KEYWORDS_DIR, RAW_PULLS_DIR, LOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------- #
# Google Trends query parameters
# --------------------------------------------------------------- #
GEO: str = "US"
START_DATE: str = "2022-01-01"
END_DATE: str = "2026-05-31"

# Google Trends silently downsamples any single request spanning
# more than ~260 days from daily to weekly resolution. 270 days
# stays just past that boundary while keeping the chunk count (and
# therefore the request count) as low as possible.
CHUNK_DAYS: int = 270

# A 30-day shared window between consecutive chunks gives the
# overlap-ratio stitching step (Section 4) enough shared history to
# estimate a stable rescaling factor between chunks.
OVERLAP_DAYS: int = 30

# --------------------------------------------------------------- #
# Adaptive pacing: cadence between *successful* requests
# --------------------------------------------------------------- #
# A short, randomized pause follows every request; a longer
# "micro-nap" happens periodically. Randomizing the interval (as
# opposed to a fixed sleep) avoids the regular request cadence that
# tends to trip Google Trends' rate limiting.
STANDARD_SLEEP_RANGE: tuple[float, float] = (12.0, 24.0)
MICRO_NAP_EVERY: int = 5
MICRO_NAP_RANGE: tuple[float, float] = (45.0, 90.0)

# --------------------------------------------------------------- #
# Retry / backoff: response to HTTP 429 and transient errors
# --------------------------------------------------------------- #
MAX_RETRIES: int = 5

# Rate-limit responses get a long, wide backoff window: once Google
# Trends starts returning 429s, backing off aggressively recovers
# faster than retrying quickly ever does.
RATE_LIMIT_BACKOFF_RANGE: tuple[float, float] = (60.0, 300.0)

# Non-rate-limit transient errors (timeouts, dropped connections)
# get a shorter window, since they are not evidence of throttling.
TRANSIENT_ERROR_SLEEP_RANGE: tuple[float, float] = (20.0, 60.0)

# (connect_timeout, read_timeout), seconds.
REQUEST_TIMEOUT: tuple[int, int] = (10, 25)

# --------------------------------------------------------------- #
# Logging
# --------------------------------------------------------------- #
LOG_FILE: Path = LOG_DIR / "collection.log"
LOG_LEVEL: str = "INFO"


## 2. Network Logic and Google Trends API Wrapper

`GoogleTrendsClient` is the only piece of code in this notebook that
talks to Google Trends.

- **Session lifecycle.** A fresh `pytrends.TrendReq` session is
  rebuilt after every rate-limit event rather than reused for the
  life of the run.
- **Adaptive pacing.** `_pace()` implements the randomized delay
  between successful requests: `base ± jitter`, with a longer
  "micro-nap" every few requests.
- **Exponential-style backoff.** `fetch_chunk()` retries on HTTP 429
  with a long, randomized backoff window, and retries transient
  network errors with a shorter one. Exceptions are mapped to
  specific `pytrends`/`requests` exception classes rather than a
  bare `except Exception`.


In [ ]:
"""Network layer: pytrends session management and adaptive pacing.
"""
import logging
import random
import time

import pandas as pd
import requests
from pytrends.exceptions import ResponseError, TooManyRequestsError
from pytrends.request import TrendReq

logger = logging.getLogger("gt_pipeline")
logger.setLevel(LOG_LEVEL)

if not logger.handlers:
    _file_handler = logging.FileHandler(LOG_FILE, encoding="utf-8")
    _stream_handler = logging.StreamHandler()
    _formatter = logging.Formatter(
        "%(asctime)s | %(levelname)-7s | %(message)s"
    )
    _file_handler.setFormatter(_formatter)
    _stream_handler.setFormatter(_formatter)
    logger.addHandler(_file_handler)
    logger.addHandler(_stream_handler)


class GoogleTrendsClient:
    """A rate-limit-aware wrapper around ``pytrends``.

    One instance is created per keyword by the orchestration layer
    (Section 3); the request counter that drives pacing is scoped to
    the client, not to a single (keyword, window) pull.
    """

    def __init__(self, request_count: int = 0) -> None:
        self._session = self._new_session()
        self._request_count = request_count

    @staticmethod
    def _new_session() -> TrendReq:
        return TrendReq(
            hl="en-US",
            tz=360,
            timeout=REQUEST_TIMEOUT,
            retries=0,
            backoff_factor=0,
        )

    def _pace(self) -> None:
        """Randomized adaptive delay between successful requests.

        A short jittered pause follows every request, with a longer
        "micro-nap" inserted periodically. Randomizing the interval
        (rather than sleeping a fixed duration) avoids presenting
        Google Trends with a mechanically regular request cadence.
        """
        self._request_count += 1
        is_micro_nap = self._request_count % MICRO_NAP_EVERY == 0
        low, high = (
            MICRO_NAP_RANGE if is_micro_nap else STANDARD_SLEEP_RANGE
        )
        delay = random.uniform(low, high)
        logger.debug(
            "%s pause: %.1fs (request #%d)",
            "Micro-nap" if is_micro_nap else "Standard",
            delay,
            self._request_count,
        )
        time.sleep(delay)

    def fetch_chunk(
        self,
        query: str,
        start: str,
        end: str,
        geo: str = GEO,
    ) -> pd.DataFrame:
        """Pull one keyword over one date window, with backoff.

        Retries up to ``MAX_RETRIES`` times. Rate-limit errors use a
        long, wide backoff window and force a fresh session; other
        transient network errors use a shorter window and reuse the
        existing session. Anything else is not retried, since it
        will not resolve on its own.
        """
        timeframe = f"{start} {end}"

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                self._session.build_payload(
                    kw_list=[query],
                    timeframe=timeframe,
                    geo=geo,
                )
                frame = self._session.interest_over_time()
                result = self._normalize(frame, query)
                self._pace()
                return result

            except TooManyRequestsError:
                if attempt == MAX_RETRIES:
                    raise
                delay = random.uniform(*RATE_LIMIT_BACKOFF_RANGE)
                logger.warning(
                    "Rate limited on %r [%s:%s], attempt %d/%d. "
                    "Backing off %.0fs and rebuilding session.",
                    query, start, end, attempt, MAX_RETRIES, delay,
                )
                time.sleep(delay)
                self._session = self._new_session()

            except (
                requests.exceptions.Timeout,
                requests.exceptions.ConnectionError,
                ResponseError,
            ) as exc:
                if attempt == MAX_RETRIES:
                    raise
                delay = random.uniform(*TRANSIENT_ERROR_SLEEP_RANGE)
                logger.warning(
                    "Transient error on %r [%s:%s], attempt %d/%d: "
                    "%s. Retrying in %.0fs.",
                    query, start, end, attempt, MAX_RETRIES, exc,
                    delay,
                )
                time.sleep(delay)

        raise RuntimeError(
            f"Exhausted retries for {query!r} over {start} to {end}."
        )

    @staticmethod
    def _normalize(
        frame: pd.DataFrame | None, query: str
    ) -> pd.DataFrame:
        """Coerce a pytrends result into a consistent two-column
        schema, and treat an empty result as a normal outcome.
        """
        if frame is None or frame.empty:
            logger.info(
                "Empty result for %r (below Trends reporting "
                "threshold for this window).",
                query,
            )
            return pd.DataFrame(columns=["Time", query])

        frame = frame.drop(columns=["isPartial"], errors="ignore")
        return frame.reset_index().rename(columns={"date": "Time"})


## 3. Pipeline Orchestration & Execution

**Checkpointing strategy.** Resumability is file-existence based,
deliberately: each `(keyword, date window)` pull is written to a
uniquely named chunk file before the next one starts, and
`run_one_query` skips any chunk file that already exists on disk.
This means the filesystem itself is the progress ledger — there is
no separate state file that can drift out of sync with what was
actually written. `run_all_queries` additionally writes a
`run_log.csv` after every keyword, giving a per-keyword audit trail
(status, error text, timestamp) independent of the chunk-level
checkpoint.

**Deduplication.** Exact duplicate query strings are pulled once,
whichever of `top` / `rising` lists they came from; cross-listed
queries are marked `rising;top` in `query_metadata.csv` rather than
silently dropped. Near-duplicates (`chatgpt` vs. `chat gpt`) are
intentionally *not* merged — see the note at the end of this
section.


In [ ]:
"""Pipeline orchestration: keyword universe, batching, and the
checkpointed run loop that drove the original keyword pull.
"""
import json
import re
from collections import defaultdict
from datetime import datetime, timedelta

import pandas as pd


# --------------------------------------------------------------- #
# Keyword universe: load, deduplicate, assign folder slugs
# --------------------------------------------------------------- #
def parse_query_block(path) -> list[str]:
    """Read one query per line from a plain-text keyword file."""
    text = path.read_text(encoding="utf-8")
    return [line.strip() for line in text.splitlines() if line.strip()]


def sanitize_folder_name(query: str) -> str:
    """Create a filesystem-safe folder slug for a query.

    Only used for the on-disk folder name — the query text sent to
    Google Trends is never altered.
    """
    name = query.strip().lower()
    name = re.sub(r'[<>:"/\\|?*]+', "", name)
    name = re.sub(r"\s+", "_", name)
    name = re.sub(r"_+", "_", name)
    name = name.strip("._ ") or "query"
    return name[:120]  # Windows MAX_PATH protection.


def build_keyword_universe(
    top_path=TOP_QUERIES_PATH,
    rising_path=RISING_QUERIES_PATH,
) -> pd.DataFrame:
    """Load, deduplicate, and slug the curated keyword lists.

    Deduplication is exact-match only (after whitespace stripping);
    see the design note in the Section 3 markdown header for why
    near-duplicates are preserved rather than merged. Queries that
    appear in both lists are pulled once and tagged
    ``source == "rising;top"`` (``sources`` is alpha-sorted).
    """
    top_queries = parse_query_block(top_path)
    rising_queries = parse_query_block(rising_path)

    sources_by_query: dict[str, set[str]] = defaultdict(set)
    for query in top_queries:
        sources_by_query[query].add("top")
    for query in rising_queries:
        sources_by_query[query].add("rising")

    # Preserve first-seen order: all `top` queries, then any
    # `rising`-only queries appended after.
    ordered_queries, seen = [], set()
    for query in top_queries + rising_queries:
        if query not in seen:
            seen.add(query)
            ordered_queries.append(query)

    used_slugs: set[str] = set()
    rows = []
    for query in ordered_queries:
        base_slug = sanitize_folder_name(query)
        slug, suffix = base_slug, 2
        while slug in used_slugs:
            slug = f"{base_slug}_{suffix}"
            suffix += 1
        used_slugs.add(slug)

        sources = sorted(sources_by_query[query])
        rows.append({
            "query": query,
            "folder": slug,
            "source": ";".join(sources),
            "in_top": "top" in sources,
            "in_rising": "rising" in sources,
        })

    return pd.DataFrame(rows)


# --------------------------------------------------------------- #
# Overlapping date windows
# --------------------------------------------------------------- #
def make_time_windows(
    start_date: str,
    end_date: str,
    chunk_days: int = CHUNK_DAYS,
    overlap_days: int = OVERLAP_DAYS,
) -> list[tuple[str, str]]:
    """Build overlapping ``(start, end)`` date-window strings.

    Windows overlap by ``overlap_days`` so that Section 4's stitcher
    has shared history to estimate a rescaling ratio between chunks.
    """
    start = pd.to_datetime(start_date).date()
    end = pd.to_datetime(end_date).date()

    windows = []
    cursor = start
    while cursor <= end:
        window_end = min(cursor + timedelta(days=chunk_days - 1), end)
        windows.append((cursor.isoformat(), window_end.isoformat()))
        if window_end >= end:
            break
        cursor = window_end - timedelta(days=overlap_days - 1)

    return windows


TIME_WINDOWS = make_time_windows(START_DATE, END_DATE)


# --------------------------------------------------------------- #
# Per-keyword run: checkpointed on chunk-file existence
# --------------------------------------------------------------- #
def write_query_metadata(row: pd.Series, query_dir) -> None:
    """Record provenance for one keyword next to its raw pulls."""
    payload = {
        "query": row["query"],
        "folder": row["folder"],
        "source": row["source"],
        "in_top": bool(row["in_top"]),
        "in_rising": bool(row["in_rising"]),
        "geo": GEO,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "chunk_days": CHUNK_DAYS,
        "overlap_days": OVERLAP_DAYS,
        "created_at": datetime.now().isoformat(timespec="seconds"),
    }
    with open(query_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)


def run_one_query(row: pd.Series, client: GoogleTrendsClient) -> None:
    """Pull every date window for one keyword, skipping any chunk
    file that already exists on disk.

    This file-existence check *is* the checkpoint: a chunk that was
    already written is trusted and never re-fetched, so a crashed
    or manually stopped run resumes for free on the next call to
    ``run_all_queries`` without any separate progress file.
    """
    query, folder = row["query"], row["folder"]
    query_dir = RAW_PULLS_DIR / folder
    chunks_dir = query_dir / "chunks"
    chunks_dir.mkdir(parents=True, exist_ok=True)
    (query_dir / "diagnostics").mkdir(exist_ok=True)
    (query_dir / "stitched").mkdir(exist_ok=True)

    write_query_metadata(row, query_dir)

    for start, end in TIME_WINDOWS:
        chunk_path = chunks_dir / f"gt_{folder}_{start}_{end}.csv"
        if chunk_path.exists():
            logger.debug("Chunk exists, skipping: %s", chunk_path.name)
            continue

        logger.info("Pulling %r: %s to %s", query, start, end)
        frame = client.fetch_chunk(query, start, end)
        frame.to_csv(chunk_path, index=False)


def run_all_queries(
    keyword_universe: pd.DataFrame,
    start_at: int = 0,
    stop_after: int | None = None,
) -> None:
    """Drive the full run, one ``GoogleTrendsClient`` per keyword.

    ``start_at`` / ``stop_after`` let a resumed run skip keywords
    that are already fully checkpointed (chunk-level resumption
    still applies within each keyword). Every keyword's outcome is
    appended to ``run_log.csv`` immediately, so the log reflects
    real progress even if the run is interrupted mid-list.
    """
    rows = keyword_universe.iloc[start_at:]
    if stop_after is not None:
        rows = rows.iloc[:stop_after]

    log_rows: list[dict] = []
    request_count = 0

    for idx, row in rows.iterrows():
        logger.info("=== [%d] %r -> %s", idx, row["query"], row["folder"])
        client = GoogleTrendsClient(request_count=request_count)

        try:
            run_one_query(row, client)
            status, error = "done", ""
        except (
            TooManyRequestsError,
            requests.exceptions.RequestException,
            ResponseError,
            RuntimeError,
        ) as exc:
            status, error = "failed", str(exc)
            logger.error("FAILED %r: %s", row["query"], error)

        request_count = client._request_count
        log_rows.append({
            "index": idx,
            "query": row["query"],
            "folder": row["folder"],
            "source": row["source"],
            "status": status,
            "error": error,
            "timestamp": datetime.now().isoformat(timespec="seconds"),
        })
        pd.DataFrame(log_rows).to_csv(RUN_LOG_PATH, index=False)

    logger.info("Run finished: %d keyword(s) processed.", len(rows))

# The original run built the keyword universe once, persisted it,
# and called `run_all_queries` in two passes: a `stop_after=2` smoke
# test to confirm Trends access and file layout, then a full,
# unbounded pass. That call is left commented out here since the
# collection phase for this project is already complete:
#
#   keyword_universe = build_keyword_universe()
#   keyword_universe.to_csv(QUERY_METADATA_PATH, index=False)
#   run_all_queries(keyword_universe)


## 4. Post-Processing: Chunk Stitching & Reassembly

Google Trends normalizes each request's 0-100 scale independently,
so adjacent chunks for the same keyword are not on a common scale
and cannot simply be concatenated. This section reassembles the
per-keyword chunk files in `chunks/` into one continuous daily
series in `stitched/`, using the 30-day overlap between consecutive
windows to estimate a rescaling factor.

**Design note — two stitching passes.** The first working version of
this step (not shown here) rescaled each new chunk against the prior
one using the *median* ratio over the full overlap window. That
approach turned out to be sensitive to shared spikes: if a keyword
had a genuine traffic spike that happened to fall inside the overlap
window, the spike dominated the ratio estimate and distorted the
rescaling for the entire chunk. The version below — the one actually
used to produce the final dataset — filters the overlap window down
to its "calm" portion (below the 95th percentile on both sides)
before computing the ratio, and separately flags any estimate that
still looks unstable (extreme scale factor, high ratio variance, low
correlation, or too little usable overlap) via `method_notes`, so
that flagged keywords can be reviewed.


In [ ]:
"""Post-processing: stitch overlapping chunks into one continuous
daily series per keyword, rescaling on the "calm" portion of each
overlap window to avoid distortion from shared traffic spikes.
"""
import numpy as np
import pandas as pd


def read_chunk_file(path, query: str) -> pd.DataFrame:
    """Load one chunk CSV into a standardized ``[Time, query]``
    frame, coercing the value column to numeric.
    """
    frame = pd.read_csv(path, parse_dates=["Time"])
    value_cols = [c for c in frame.columns if c != "Time"]
    if len(value_cols) != 1:
        raise ValueError(
            f"{path.name} has {len(value_cols)} value columns: "
            f"{value_cols}"
        )

    frame = (
        frame[["Time", value_cols[0]]]
        .rename(columns={value_cols[0]: query})
        .sort_values("Time")
        .drop_duplicates("Time", keep="first")
        .reset_index(drop=True)
    )
    frame[query] = pd.to_numeric(frame[query], errors="coerce")
    return frame


def estimate_overlap_scale(
    base: pd.DataFrame,
    new: pd.DataFrame,
    query: str,
    min_overlap_days: int = 10,
    spike_pctile: float = 95.0,
    min_reasonable_scale: float = 0.2,
    max_reasonable_scale: float = 5.0,
    max_ratio_cv: float = 1.0,
    min_overlap_corr: float = 0.3,
) -> dict:
    """Estimate the factor that rescales ``new`` onto ``base``'s
    Trends-index scale, using only the "calm" (sub-``spike_pctile``)
    portion of the overlap window when enough calm days exist.

    Returns a diagnostics dict including a ``method_notes`` field
    that flags scale estimates worth a manual second look, rather
    than only the estimate itself.
    """
    overlap = pd.merge(
        base[["Time", query]].rename(columns={query: "base_value"}),
        new[["Time", query]].rename(columns={query: "new_value"}),
        on="Time",
        how="inner",
    ).dropna(subset=["base_value", "new_value"])

    valid = overlap[
        (overlap["base_value"] > 0) & (overlap["new_value"] > 0)
    ].copy()
    valid_before_spike_filter = len(valid)

    spike_filter_used = False
    valid_for_ratio = valid
    if len(valid) >= min_overlap_days:
        base_thresh = valid["base_value"].quantile(spike_pctile / 100)
        new_thresh = valid["new_value"].quantile(spike_pctile / 100)
        calm = valid[
            (valid["base_value"] <= base_thresh)
            & (valid["new_value"] <= new_thresh)
        ]
        if len(calm) >= min_overlap_days:
            valid_for_ratio = calm
            spike_filter_used = True

    has_variance = (
        len(overlap) >= 2
        and overlap["base_value"].std() > 0
        and overlap["new_value"].std() > 0
    )
    corr = overlap["base_value"].corr(overlap["new_value"]) \
        if has_variance else np.nan

    if len(valid_for_ratio) >= min_overlap_days:
        ratio = valid_for_ratio["new_value"] / valid_for_ratio["base_value"]
        median_ratio = ratio.median()
        scale_new_to_base = (
            1 / median_ratio
            if median_ratio != 0 and pd.notna(median_ratio)
            else 1.0
        )
        ratio_mean = ratio.mean()
        ratio_cv = (
            ratio.std() / ratio_mean
            if ratio_mean != 0 and pd.notna(ratio_mean)
            else np.nan
        )
        method = (
            "overlap_ratio_spike_filtered"
            if spike_filter_used
            else "overlap_ratio"
        )
    else:
        median_ratio, ratio_cv = np.nan, np.nan
        scale_new_to_base = 1.0
        method = "no_rescale_insufficient_overlap"

    notes = []
    if pd.notna(scale_new_to_base) and not (
        min_reasonable_scale <= scale_new_to_base <= max_reasonable_scale
    ):
        notes.append("extreme_scale_factor")
    if pd.notna(ratio_cv) and ratio_cv > max_ratio_cv:
        notes.append("unstable_overlap_ratio")
    if pd.notna(corr) and corr < min_overlap_corr:
        notes.append("low_overlap_corr")
    if len(valid_for_ratio) < min_overlap_days:
        notes.append("insufficient_valid_overlap")

    return {
        "overlap_days": len(overlap),
        "valid_ratio_days_before_spike_filter": valid_before_spike_filter,
        "valid_ratio_days": len(valid_for_ratio),
        "spike_pctile": spike_pctile,
        "spike_filter_used": spike_filter_used,
        "overlap_corr": corr,
        "median_ratio_new_over_base": median_ratio,
        "scale_new_to_base": scale_new_to_base,
        "ratio_cv": ratio_cv,
        "method": method,
        "method_notes": ";".join(notes),
    }


def stitch_chunks_overlap_ratio(
    chunk_files: list,
    query: str,
    min_overlap_days: int = 10,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Fold a sorted list of chunk files into one rescaled, deduped
    daily series, returning the series and a per-step diagnostics
    table produced by ``estimate_overlap_scale``.
    """
    chunk_files = sorted(chunk_files)
    if not chunk_files:
        return pd.DataFrame(columns=["Time", query]), pd.DataFrame()

    stitched = read_chunk_file(chunk_files[0], query)
    diag_rows = [{
        "step": 0,
        "file": chunk_files[0].name,
        "scale_new_to_base": 1.0,
        "method": "base_chunk",
    }]

    for step, path in enumerate(chunk_files[1:], start=1):
        new = read_chunk_file(path, query)
        scale_info = estimate_overlap_scale(
            base=stitched,
            new=new,
            query=query,
            min_overlap_days=min_overlap_days,
        )
        new[query] = new[query] * scale_info["scale_new_to_base"]

        stitched = (
            pd.concat([stitched, new], ignore_index=True)
            .groupby("Time", as_index=False)[query]
            .mean()
            .sort_values("Time")
            .reset_index(drop=True)
        )
        diag_rows.append({"step": step, "file": path.name, **scale_info})

    return stitched, pd.DataFrame(diag_rows)


def stitch_one_query_folder(query_dir) -> pd.DataFrame:
    """Stitch a single keyword's chunk files and write the result
    and its diagnostics under that keyword's directory.
    """
    folder = query_dir.name
    (query_dir / "diagnostics").mkdir(exist_ok=True)
    (query_dir / "stitched").mkdir(exist_ok=True)

    chunk_files = sorted((query_dir / "chunks").glob("gt_*.csv"))
    stitched, scale_diag = stitch_chunks_overlap_ratio(
        chunk_files=chunk_files, query=folder, min_overlap_days=10,
    )

    stitched_path = (
        query_dir / "stitched"
        / f"gt_stitched_{folder}_{START_DATE}_{END_DATE}.csv"
    )
    stitched.to_csv(stitched_path, index=False)

    scale_diag = scale_diag.copy()
    scale_diag.insert(0, "query", folder)
    scale_diag.to_csv(
        query_dir / "diagnostics" / "stitching_diagnostics.csv",
        index=False,
    )
    logger.info("Stitched %s: %d rows -> %s", folder, len(stitched),
                stitched_path)
    return scale_diag

# The original run stitched every keyword folder under
# `RAW_PULLS_DIR` and concatenated the per-keyword diagnostics into
# one master QA table. Left commented out here since the collection
# and post-processing phases are already complete.
#
#   all_diagnostics = [
#       stitch_one_query_folder(query_dir)
#       for query_dir in sorted(RAW_PULLS_DIR.iterdir())
#       if query_dir.is_dir() and (query_dir / "chunks").exists()
#   ]
#   pd.concat(all_diagnostics, ignore_index=True).sort_values(
#       ["query", "step"]
#   ).to_csv(SCALE_DIAGNOSTICS_PATH, index=False)
